In [8]:
# ============================================
# 配置区 —— 只改这里就够了
# ============================================
BOOK_ID = 1      # 书卷 ID
CHAPTER = 1      # 章节

DB_PATH = "db/bible.db"                     # 数据库路径
OUT_DIR = "static/script"                   # 输出目录

# ============================================
# 初始化
# ============================================
import sqlite3
import json
import os

os.makedirs(OUT_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# ============================================
# SQL：按 book_id + chapter 读取 tokens
# ============================================
sql = """
SELECT
    v.verse,
    v.text_cn,
    t.token,
    t.type,
    t.entity_key,
    t.align_id,
    t.token_id,
    ts.start_time,
    ts.end_time
FROM verses v
LEFT JOIN tokens t
    ON v.book_id = t.book_id
   AND v.chapter = t.chapter
   AND v.verse = t.verse
LEFT JOIN timestamps ts
    ON t.align_id = ts.id
WHERE v.book_id = ?
  AND v.chapter = ?
ORDER BY v.verse, t.token_id
"""

rows = cur.execute(sql, (BOOK_ID, CHAPTER)).fetchall()
conn.close()

# ============================================
# 组装 JSON（按 verse 分组）
# ============================================
result = []
current_verse = None
verse_obj = None

for r in rows:
    if current_verse != r["verse"]:
        verse_obj = {
            "verse": r["verse"],
            "text_cn": r["text_c"],
            "words": []
        }
        result.append(verse_obj)
        current_verse = r["verse"]

    if r["token"]:
        verse_obj["words"].append({
            "word": r["token"],
            "type": r["type"] or "",
            "entity_key": r["entity_key"] or "",
            "align_id": r["align_id"] or "",
            "start": r["start_time"] or 0,
            "end": r["end_time"] or 0
        })

# ============================================
# 输出 JSON 文件
# ============================================
out_file = f"{BOOK_ID}_{CHAPTER}.json"
out_path = os.path.join(OUT_DIR, out_file)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"✅ 导出成功：{out_path}")
print(f"📖 书卷 ID: {BOOK_ID}, 章节: {CHAPTER}")
print(f"📦 总节数: {len(result)}")

IndexError: No item with that key

还在跑
